Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [36]:
NO_RECORDS = 150

In [37]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

In [38]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [39]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [40]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [41]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/127769196', 'title': 'Calibration', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/127769196</identifier>\n    <datestamp>2025-01-08T11:59:52.308Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/22/facilityCycle/127395649/investigation/127769196</identifier>\n          <alternateIdentifi

In [42]:
record_titles

['Calibration',
 'Calibration',
 'Instrument Calibration',
 'mSR study of charge carrier motion in thermally reduced graphene oxide',
 "PSI collaboration tests (Craig Bull's team)",
 'Studying the spin asymmetry of magnetic nanoparticles',
 'ChipIr beam profile measurements',
 'Calibration',
 'Calib of Instrument',
 'INTER Calibration cycle 18_3',
 'Neutron Training Course Practical',
 'Phonons in VO2. ISIS NTC',
 'Inelastic measurement of the battery electrolyte PC:CaTFSI and its solvent PC',
 'Calibration',
 'µSR study of the density wave state in a-(BEDT-TTF)2MHg(SCN)4 (M=K, Rb)',
 'g',
 'Commissioning condensation cells',
 'Calibration Scans',
 'RF cryostat',
 'NRCA set-up',
 'Magnetism of an insulator phase near high-Tc honeycomb lattice superconducting phase',
 'Complexity in simplicity: disentangling Pb2+ disorder in PbS, PbSe and PbTe',
 'Development of the in-situ Electronic-Filed Application µSR Technique and Test Application to Multiferroic Systems.',
 'Super-MuSR detector t

In [43]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/111244144</identifier>\n    <datestamp>2020-11-10T09:47:48.372Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/27/facilityCycle//investigation/111244144</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>2035508</alternateIdentifier>\n          </a

In [44]:
get_doi(record_raw_metadata[2])

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [45]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [46]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

NO DOI


In [47]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[3, 14, 20, 21, 22, 25, 40, 41, 43, 50, 51, 60, 69, 76, 80, 93, 99, 100, 101, 105, 106, 110, 111, 122, 126, 147]


In [48]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/63838079', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773725', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/59895536', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/85068924', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224162', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/82439956', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/79637488', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/61011265', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/42588597', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/117414098', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/49911793', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/42591854', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/70307814', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/83551804', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/54816815', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773642', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/95666101', '0725037e-553a-4a96-ac2b

In [49]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [50]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Mr Mark Kibble', 'name': 'Mark', 'surname': 'Kibble', 'rank': 1, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'RLI838'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'RLI904'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'PG002'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'Technique Development'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'cell'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': '+'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'Construction Engineering and Architecture'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'RLI908'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'Sine2020'}, 'provenance': None}, {'subject': {'schem

In [51]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.5286/isis.e.rb1790346-1',
  'subjects': ['RLI838',
   'RLI904',
   'PG002',
   'Technique Development',
   'cell',
   '+',
   'Construction Engineering and Architecture',
   'RLI908',
   'Sine2020',
   'Cd',
   'RAVIL',
   'Engineering',
   'RLI882',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-6',
  'subjects': ['RLI838',
   'RLI904',
   'PG002',
   'Technique Development',
   'cell',
   '+',
   'Construction Engineering and Architecture',
   'RLI908',
   'Sine2020',
   'Cd',
   'RAVIL',
   'Engineering',
   'RLI882',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-7',
  'subjects': ['RLI838',
   'RLI904',
   'PG002',
   'Technique Development',
   'cell',
   '+',
   'Construction Engineering and Architecture',
   'RLI908',
   'Sine2020',
   'Cd',
   'RAVIL',
   'Engineering',
   'RLI882',
   'Engineering Sciences',
   'Materials',

In [52]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': 'https://doi.org/10.5286/isis.e.rb1670547-3'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/63838079',
  'doi': 'https://doi.org/10.5286/isis.e.63838079'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773725',
  'doi': 'https://doi.org/10.5286/isis.e.67773725'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/59895536',
  'doi': 'https://doi.org/10.5286/isis.e.59895536'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/85068924',
  'doi': 'https://doi.org/10.5286/isis.e.85068924'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224162',
  'doi': 'https://doi.org/10.5286/isis.e.rb1790346-3'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/82439956',
  'doi': 'https://doi.org/10.5286/isis.e.82439956'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/79637488',
  'doi': 'https://doi.org/10.5286/isis.e.79637488'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/61011265',
  'doi': '

In [53]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': 'https://doi.org/10.5286/isis.e.rb1670547-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/63838079',
  'doi': 'https://doi.org/10.5286/isis.e.63838079',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773725',
  'doi': 'https://doi.org/10.5286/isis.e.67773725',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/59895536',
  'doi': 'https://doi.org/10.5286/isis.e.59895536',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/85068924',
  'doi': 'https://doi.org/10.5286/isis.e.85068924',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224162',
  'doi': 'https://doi.org/10.5286/isis.e.rb1790346-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/82439956',
  'doi': 'https://doi.org/10.5286/isis.e.82439956',
  'enric

In [54]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': 'https://doi.org/10.5286/isis.e.rb1670547-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/63838079',
  'doi': 'https://doi.org/10.5286/isis.e.63838079',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773725',
  'doi': 'https://doi.org/10.5286/isis.e.67773725',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/59895536',
  'doi': 'https://doi.org/10.5286/isis.e.59895536',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/85068924',
  'doi': 'https://doi.org/10.5286/isis.e.85068924',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224162',
  'doi': 'https://doi.org/10.5286/isis.e.rb1790346-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/82439956',
  'doi': 'https://doi.org/10.5286/isis.e.82439956',
  'enric

In [55]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': 'https://doi.org/10.5286/isis.e.rb1670547-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/63838079',
  'doi': 'https://doi.org/10.5286/isis.e.63838079',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773725',
  'doi': 'https://doi.org/10.5286/isis.e.67773725',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/59895536',
  'doi': 'https://doi.org/10.5286/isis.e.59895536',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/85068924',
  'doi': 'https://doi.org/10.5286/isis.e.85068924',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224162',
  'doi': 'https://doi.org/10.5286/isis.e.rb1790346-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/82439956',
  'doi': 'https://doi.org/10.5286/isis.e.82439956',
  'enric

In [56]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

26
26


In [57]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [58]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[3, 14, 20, 21, 22, 25, 40, 41, 43, 50]

In [59]:
first_enriched

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': 'https://doi.org/10.5286/isis.e.rb1670547-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/63838079',
  'doi': 'https://doi.org/10.5286/isis.e.63838079',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/67773725',
  'doi': 'https://doi.org/10.5286/isis.e.67773725',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/59895536',
  'doi': 'https://doi.org/10.5286/isis.e.59895536',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/85068924',
  'doi': 'https://doi.org/10.5286/isis.e.85068924',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224162',
  'doi': 'https://doi.org/10.5286/isis.e.rb1790346-3',
  'enriched_subjects': set()},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/82439956',
  'doi': 'https://doi.org/10.5286/isis.e.82439956',
  'enric

In [60]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/105604808</identifier>\n    <datestamp>2019-08-12T11:56:24.663Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB1670547-3</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>1670547</alternateIdentifier>\n     

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [61]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [62]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [63]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [64]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

0


[]

In [65]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.7
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [66]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [67]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

# Related DOIs  
Extract related identifiers that are also DOIs. If they are in the database, use their subjects. If not, use DOI-resolving methods on them.

Note:
a DOI must start with 10. prefix. I've seen some related DOIs that start with 0.  , if I add a 1 to the start it resolves properly.

## SQL query to see records with related DOIs   
```
--- get records with related identifiers of any tipe
WITH query AS (
    SELECT
        raw_metadata,
		doi as record_doi,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 1000;
```

47k records have SOME relatedIdentifiers - lots of them ISSN or proprietary ids